In [ ]:
import cdsapi
from pathlib import Path
import xarray as xr, numpy as np
import subprocess, sys


# Monthly CDS download

In [ ]:
c = cdsapi.Client()
out = Path("data/era5_monthly"); out.mkdir(parents=True, exist_ok=True)

AREA = [44.0, -10.0, 36.25, 3.25]   # N, W, S, E -> 32 lat x 54 lon
GRID = [0.25, 0.25]

for year in range(2000, 2026):
    temperature_target = out / f"era5_monthly_temperature_{year}.nc"
    precipitation_target = out / f"era5_monthly_precipitation_{year}.nc"
    
    if temperature_target.exists() and precipitation_target.exists():
        print(f"skip {year}"); continue
    try:
        c.retrieve(
            "reanalysis-era5-single-levels-monthly-means",
            {
                "product_type": "monthly_averaged_reanalysis",
                "variable": ["2m_temperature"],
                "year": str(year),
                "month": [f"{m:02d}" for m in range(1, 13)],
                "time": "00:00",
                "area": AREA,
                "grid": GRID,
                "data_format": "netcdf",
            },
            str(temperature_target),
        )
        c.retrieve(
            "reanalysis-era5-single-levels-monthly-means",
            {
                "product_type": "monthly_averaged_reanalysis",
                "variable": ["total_precipitation"],
                "year": str(year),
                "month": [f"{m:02d}" for m in range(1, 13)],
                "time": "00:00",
                "area": AREA,
                "grid": GRID,
                "data_format": "netcdf",
            },
            str(precipitation_target),
        )

    except Exception as e:
        print(f"FAILED {year}: {e}")

# Hourly CDS download

In [ ]:
c = cdsapi.Client()
out = Path("data/era5_3hourly"); out.mkdir(parents=True, exist_ok=True)

AREA = [44.0, -10.0, 36.25, 3.25]
GRID = [0.25, 0.25]
TIMES = ["00:00", "03:00", "06:00", "09:00",
         "12:00", "15:00", "18:00", "21:00"]
DAYS  = [f"{d:02d}" for d in range(1, 32)]
VARS  = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]

for year in range(2000, 2026):
    for month in range(1, 13):
        target = out / f"era5_3h_surface_{year}_{month:02d}.nc"
        if target.exists():
            continue
        try:
            c.retrieve(
                "reanalysis-era5-single-levels",
                {
                    "product_type": "reanalysis",
                    "variable": VARS,
                    "year": str(year),
                    "month": f"{month:02d}",
                    "day": DAYS,
                    "time": TIMES,
                    "area": AREA,
                    "grid": GRID,
                    "data_format": "netcdf",
                },
                str(target),
            )
        except Exception as e:
            print(f"FAILED {year}-{month:02d}: {e}")

# CDS static land-sea mask

In [ ]:
c = cdsapi.Client()
out = Path("data/era5_static"); out.mkdir(parents=True, exist_ok=True)

AREA = [44.0, -10.0, 36.25, 3.25]
GRID = [0.25, 0.25]

target = out / "era5_lsm.nc"
if not target.exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": "land_sea_mask",
            "year": "2020",
            "month": "01",
            "day": "01",
            "time": "00:00",
            "area": AREA,
            "grid": GRID,
            "data_format": "netcdf",
        },
        str(target),
    )

# Preprocess hourly data to calculate VPD

In [ ]:
def es(T_C):  # Tetens, hPa
    return 6.1078 * np.exp(17.27 * T_C / (T_C + 237.3))

def process_one(path):
    with xr.open_dataset(path) as ds:
        ds = ds.load()
        if "valid_time" in ds.coords:
            ds = ds.rename({"valid_time": "time"})
        vpd = es(ds["t2m"] - 273.15) - es(ds["d2m"] - 273.15)
        ws  = np.sqrt(ds["u10"]**2 + ds["v10"]**2)
        out = xr.Dataset({
            "vpd_monthly_mean_daymax":
                vpd.resample(time="1D").max().resample(time="1MS").mean(),
            "wind_speed_monthly":
                ws.resample(time="1MS").mean(),
        })
        return out.load()

files = sorted(Path("data/era5_3hourly").glob("era5_3h_surface_*.nc"))
monthly = xr.concat([process_one(f) for f in files], dim="time").sortby("time")
monthly = monthly.rename({"time": "valid_time"})
monthly.to_netcdf("data/era5_derived_monthly.nc")